In [2]:
import os
from dotenv import load_dotenv

In [13]:
load_dotenv(override=True)

ollamaApiKey = os.getenv("OLLAMA_API_KEY")
ollamaBaseUrl = os.getenv("OLLAMA_BASE_URL")
print(ollamaApiKey, ollamaBaseUrl)


random_value http://localhost:11434/v1


In [6]:
DEEPSEEK_MODEL = "deepseek-r1:8b"
QWEN_MODEL = "qwen3:14b"
GEMMA_MODEL = "gemma3:12b"
GPT_MODEL = "gpt-oss:20b"

In [9]:
def generateBody(question):
    return [{"role": "user", "content": question}]


In [11]:
question = "Give me a question that multiple AI models can answer. I want to use the answers from multiple models and score them based on the response they generate. Give me the question only, no answers."
body = generateBody(question=question)
print(body)

[{'role': 'user', 'content': 'Give me a question that multiple AI models can answer. I want to use the answers from multiple models and score them based on the response they generate. Give me the question only, no answers.'}]


In [12]:
from openai import OpenAI

In [14]:
openAiClient = OpenAI(base_url=ollamaBaseUrl, api_key=ollamaApiKey)

In [17]:
def generateAnswer(model, body):
    response = openAiClient.chat.completions.create(
        model=model,
        messages=body
    )
    return response.choices[0].message.content

In [ ]:
questionToBeAskedToLLM = generateAnswer(model=QWEN_MODEL, body=body)


In [36]:
print(questionToBeAskedToLLM)

<think>
Okay, the user wants a question that multiple AI models can answer, and they want the answers to be scored based on the responses. Let me think about the requirements here.

First, the question needs to be one that various models can tackle. It shouldn't be too obscure or require specific training data that only a few models have. Maybe a question that tests general knowledge, reasoning, or creativity. Also, the answer should have a clear right or wrong aspect, but perhaps with room for different approaches, which would make scoring easier.

I need to consider the scoring aspect. If the question is too open-ended, scoring might be subjective. But if it's a question where there's a commonly accepted correct answer, or where multiple valid answers exist but can be evaluated on accuracy, reasoning, or completeness, that might work better.

Maybe a question that involves multiple steps of reasoning. For example, something from mathematics, science, or a well-known problem. Let's th

In [20]:
def parseResponseByRemovingTag(tagName, blob):
    from bs4 import BeautifulSoup
    parsedQuestion = BeautifulSoup(blob, "html.parser")
    thinkTag = parsedQuestion.find(tagName)
    if thinkTag:
        thinkTag.decompose()
    return parsedQuestion.prettify()

In [37]:
parsedQuestionToBeAskedToLLM = parseResponseByRemovingTag("think", questionToBeAskedToLLM)
print(parsedQuestionToBeAskedToLLM)

What are the key differences between machine learning and deep learning, and in what types of problems is each approach typically more effective?



In [24]:
bodyForNextLLMs = generateBody(parsedQuestionToBeAskedToLLM)
print(bodyForNextLLMs)

[{'role': 'user', 'content': 'What are the key differences between machine learning and deep learning, and in what types of problems is each approach typically more effective?\n'}]


In [ ]:
from IPython.display import Markdown, display

def parseMarkdown(blob):
    return Markdown(blob)

In [25]:
gemmaAnswer = generateAnswer(model=GEMMA_MODEL, body=bodyForNextLLMs)

In [29]:
print(display(parseMarkdown(gemmaAnswer)))

Okay, let's break down the key differences between machine learning (ML) and deep learning (DL), and then discuss their effectiveness in different problem types.

**1. Machine Learning (ML): The Broad Concept**

* **What it is:** Machine learning is a field of computer science that enables computers to learn from data without being explicitly programmed. It's about creating algorithms that can identify patterns, make predictions, and make decisions based on data.  The algorithm is "trained" on data and then can be used to make predictions on new, unseen data.
* **How it works (typically):**
    * **Feature Engineering:**  A critical step in traditional ML.  This involves humans analyzing the data and identifying the *features* (or variables) that are most relevant for the algorithm to learn from.  For example, if you're predicting house prices, features might include square footage, number of bedrooms, location, etc.  The quality of the features *heavily* influences the performance of the algorithm.
    * **Algorithm Selection:** Choosing from a toolbox of algorithms (e.g., linear regression, logistic regression, decision trees, support vector machines, k-means clustering).
    * **Model Training:** The algorithm learns the relationship between the features and the target variable (the thing you're trying to predict).
    * **Model Evaluation:** Testing the model on unseen data to assess its accuracy and generalization ability.  Tuning the model (e.g., adjusting hyperparameters) based on the evaluation results.
* **Examples:**
    * **Spam filtering:** Identifying emails as spam or not spam.
    * **Credit risk assessment:** Predicting the likelihood of a loan applicant defaulting.
    * **Recommendation systems (basic):** Suggesting products based on past purchases.
    * **Fraud detection:** Identifying suspicious transactions.

**2. Deep Learning (DL): A Subset of Machine Learning**

* **What it is:** Deep learning is a *subfield* of machine learning that utilizes artificial neural networks with *multiple layers* (hence “deep”). These networks are inspired by the structure and function of the human brain.
* **How it works:**
    * **No or Minimal Feature Engineering:** This is a HUGE difference.  Deep learning models *automatically* learn features from the data.  The neural network's layers progressively extract higher-level and more abstract features. You don't need to explicitly tell the network which features are important; it figures it out.
    * **Neural Networks:** Built from interconnected nodes (neurons) organized in layers.
        * **Input Layer:** Receives the raw data.
        * **Hidden Layers:** Multiple layers where complex feature extraction occurs.
        * **Output Layer:** Produces the prediction.
    * **Backpropagation:** A core algorithm used to train deep learning models.  It calculates the error between the model's predictions and the actual values, and then propagates this error back through the network, adjusting the connections between neurons to improve accuracy.
* **Examples:**
    * **Image Recognition:** Identifying objects in images (e.g., faces, cars, animals).  Driving self-driving cars relies heavily on this.
    * **Natural Language Processing (NLP):** Understanding and generating human language. (e.g., chatbots, translation, sentiment analysis).
    * **Speech Recognition:** Converting spoken words into text (e.g., Siri, Alexa).
    * **Generating Art and Music:** AI models can now create original works of art and music.

**Key Differences Summarized in a Table:**

| Feature           | Machine Learning (ML) | Deep Learning (DL) |
|--------------------|-----------------------|-----------------------|
| **Architecture**  | Various (decision trees, SVMs, linear regression, etc.) | Artificial Neural Networks (ANNs) with many layers |
| **Feature Engineering** | Required – Manual process | Automatic – Learns features from data |
| **Data Requirements** | Typically works with smaller datasets | Requires *massive* amounts of data to train effectively |
| **Computational Resources** | Less demanding | Very computationally expensive (GPUs often required) |
| **Interpretability** | Generally more interpretable (you can often understand *why* a model made a certain prediction) | Typically a "black box" – hard to understand the internal workings |
| **Training Time** | Faster | Significantly slower |
| **Problem Complexity** | Best for simpler problems and when you have domain expertise |  Excels at highly complex problems |



**When to Use Each Approach:  Problem Suitability**

* **Machine Learning (ML) - Good for:**
    * **Smaller Datasets:** ML algorithms can perform relatively well even with limited data.
    * **Structured Data:**  Data that is organized in rows and columns (e.g., spreadsheets, databases).
    * **Problems where Interpretability is Important:** If you need to understand *why* a model made a decision (e.g., medical diagnosis), ML is often preferred.
    * **Well-defined Features:** When you have expertise in the problem domain and can identify the relevant features.
    * **Examples:** Credit scoring, predicting customer churn, simple fraud detection, basic recommendation systems.

* **Deep Learning (DL) - Good for:**
    * **Large Datasets:**  Deep learning models need lots and lots of data to learn complex patterns.
    * **Unstructured Data:**  Data that isn't easily organized into rows and columns (e.g., images, audio, text).
    * **Complex Problems:** When the relationships between inputs and outputs are extremely complicated and difficult to define manually.
    * **Automatic Feature Extraction is Desired:** When you don't want to rely on human expertise to identify features.
    * **Examples:** Image recognition, natural language processing, speech recognition, self-driving cars, advanced drug discovery.

**Think of it this way:**

* **ML is like teaching a child to identify types of birds based on the features you tell them:** "Look, this bird has a red head, a yellow belly, and a long beak - it's a cardinal!"
* **DL is like showing a child a massive collection of bird images and letting them figure out the characteristics of different bird types on their own.**

**Important Considerations:**

* **It's not always an either/or situation:** Sometimes, a combination of ML and DL techniques can be beneficial.
* **Deep learning isn't always the best solution:**  It's more complex to implement and requires significantly more resources.  For some problems, a simpler ML algorithm might be more effective.  Don’t automatically jump to deep learning - start with simpler models first and only move to deep learning if necessary.



I hope this comprehensive explanation clarifies the differences and applications of machine learning and deep learning! Let me know if you have any further questions.

None


In [30]:
deepseekAnswer = generateAnswer(model=DEEPSEEK_MODEL, body=bodyForNextLLMs)

In [31]:
deepseekAnswerCleaned = parseResponseByRemovingTag("think", deepseekAnswer)
print(display(parseMarkdown(deepseekAnswerCleaned)))

Okay, let's break down the key differences between Machine Learning (ML) and Deep Learning (DL). While there are nuances, understanding this core relationship helps clarify things:

| **Criteria** | **Machine Learning (ML)** | **Deep Learning (DL)** |
|--------------|--------------------------|------------------------|
| **Foundation** | Set of algorithms that enable computer systems to learn from data. Hugely encompasses both shallow and deep approaches. | A subset/Specialization of ML, specifically based on artificial neural networks with multiple layers ("deep"). Usually implies multi-layered neural networks (artificial neural nets). |
| **Complexity/Depth** | Relies on handcrafted features: domain experts define the characteristics the computer should pay attention to in order to learn. Algorithms typically work well when these explicit feature representations exist, and their design can often be simplified explicitly as "what rules it learns". | Relies on automatically learning hierarchical feature representations from raw data through deep neural networks (layers). The system uncovers more complex patterns by composing simpler ones. |
| **Algorithm Type** | A wide variety of algorithms: K-Nearest Neighbors, Support Vector Machines, Decision Trees ("sklearn" area), Gaussian Naive Bayes etc., also simple Neural Networks are considered ML if rule-counting dominates the learning process (especially shallower models). | Primarily uses multi-layered artificial neural networks that can learn to perform tasks such as image recognition or natural language processing without being explicitly programmed. |
| **Data Format** | Tends to require structured, tabular formats where data points correspond clearly to columns/rows with known relationships between features and target. | Excel at handling unstructured data like text (sequences), images (2D grids/pixels), audio (1D signals). These natural structures can be more effectively captured by layers within a neural network structure. |
| **Data Dependency** | Can often work well even with relatively smaller datasets, especially if the features are good and/or the problem is simple or interpretable in terms of rules. However, some complex ML models might require decent amounts of data too! | Requires significantly more training examples (unless transfer learning from a pre-trained model helps). Without sufficient data to detect patterns across many layers/weights, it won't learn effectively without overfitting. |
| **Feature Engineering** | Explicitly required: often involves transforming raw variables into features relevant for the task (e.g., text length or image pixel intensities) before feeding them to simpler ML algorithms. | Can largely automate feature engineering; deep learning models can automatically create hierarchical, useful representations directly from raw inputs like images pixels ("autoencoder" concept), audio waveforms, or sentences ("embedding"). |
| **Computational Resources** | Relies on algorithms with relatively clear, efficient relationships between input features and output predictions (unless SVMs are supported by kernels). Training is typically faster for standard algorithms. Can be optimized well using specific techniques depending on the data format. | Relies heavily on complex math of multiple dimensions (neurons interconnecting through layers); requires intensive computation power. GPUs/TPUs necessary due to high dimensionality and numerous coefficients (weights) involved during training, especially for large models. Training can take from minutes to days/weeks/billions potentially even months depending type of DL model and hardware used! |
| **Flexibility/Hyperparameters** | Less complex models have fewer hyperparameter choices or optimizations exist for common use cases ("tuned" vs "default"). Can have interpretable models (output decisions based on the rules it learned). | More flexible but computationally expensive. Has many more layers, node types, and weights per layer to tune; process is often trial-and-error requiring significant tuning expertise. Typically harder to interpret after training; requires understanding of gradient descent/back-propagation etc. |
| **Key Examples** | Traditional algorithms for regression/"deep learning" supervised classification (like logistic/linear回归), decision tree models, linear discriminant analysis, Gaussian Naive Bayes text classifiers when features are engineered well. | Image recognition ("CNNs"), natural language processing ("RNNs/LSTMs", Transformers like GPT-3), voice recognition ("RNN Transducers"), complex pattern prediction from large datasets and generating content (text, images) using "Transformers" which are multi-layer DL models. |

To summarize the relationship:

Deep Learning is **a subset** of Machine Learning that specifically refers to systems based on **artificial neural networks with many layers**, used for tasks where shallow methods failed.

Consider Deep learning appropriate if you encounter situations such as:
1.  Large amounts of data (especially unstructured like text, audio, images).
2.  Tasks involving complex patterns recognition or understanding.
3.  Need to generate content or make predictions from raw inputs without extensive feature engineering.
4.  Existing features are unclear or non-existent for the problem type.

Choose Machine Learning first if:
1.  The data is small in quantity relative to what deep learning requires (unless transfer learning applies).
2.  The data lacks structure (or has good structured data suitable for ML models like decision trees/linear models).
3.  You need interpretability from a simpler model.
4.  Resources or expertise limit the use of complex neural networks architectures.


None


In [33]:
gptAnswer = generateAnswer(GPT_MODEL, bodyForNextLLMs)

In [34]:
print(display(parseMarkdown(gptAnswer)))

### 1. What **machine learning** (ML) actually is  

| Characteristic | Classic ML (SVM, Random Forest, Gradient‑Boosted Trees, Logistic Regression, K‑NN, …) |
|-----------------|----------------------------------------------------------------------------|
| **Model family** | Shallow, hand‑crafted models or ensembles of simple learners. |
| **Feature engineering** | **Explicit**: a human (or a rule‑based system) writes the features that will be fed to the learner. |
| **Representation** | Usually a fixed‑length vector (e.g., 100–1 000 features). |
| **Interpretability** | Often highly interpretable (feature importance, decision rules, linear coefficients). |
| **Input type** | Structured tables (tabular data) or very small dimensional data (< 100‑s of features). |
| **Training resources** | CPU‑friendly, quick to train, scales well to tens of millions of instances. |
| **Typical use‑cases** | Credit scoring, fraud detection, recommendation, simple image classification, text classification (when you have engineered TF‑IDF or word‑counts), tabular medical diagnostics, sensor data, etc. |

---

### 2. What **deep learning** (DL) actually is  

| Characteristic | Deep Neural Networks (CNNs, RNNs, Transformers, Graph Nets, etc.) |
|-----------------|-------------------------------------------------------------------|
| **Model family** | Deep hierarchical neural networks with many layers (tens to hundreds). |
| **Feature engineering** | **Implicit**: the network learns its own feature hierarchy from raw data. |
| **Representation** | Learned, high‑dimensional embeddings; end‑to‑end mapping from raw signal to label. |
| **Interpretability** | Typically opaque “black‑box,” but tools exist (saliency, feature visualization). |
| **Input type** | Raw high‑dimensional signals: pixels, audio waveforms, text tokens, point clouds, graphs. |
| **Training resources** | GPU‑ or TPU‑heavy, long training times, large memory footprints. |
| **Typical use‑cases** | Image recognition, object detection, speech recognition, machine translation, generative modeling, self‑driving cars, deep RL, large scale recommendation, anomaly detection on unstructured data. |

---

### 3. Key technical differences  

| Dimension | Classic ML | Deep Learning |
|-----------|------------|---------------|
| **Learning paradigm** | *Shallow* parameter sets (1–3 hidden layers or none). | *Deep* multi‑layered transformations. |
| **Feature learning** | Hand‑crafted, domain‑specific. | Automatic, representation‑learning. |
| **Data dependence** | Works well with *small to moderate* labeled data. | Needs *large* labeled corpora to avoid overfitting. |
| **Generalization** | Often more robust to noisy or poorly represented features. | Requires careful regularization, data augmentation, and a rich training set. |
| **Scalability (model size)** | Linear or near‑linear in number of parameters and data. | Sub‑quadratic (due to GPU parallelism) but still memory‑bound. |
| **Optimization** | Typically convex or mildly non‑convex, easier to optimize. | Highly non‑convex, gradient‑based, many local minima, needs careful initialization, learning‑rate schedules, etc. |
| **Explainability** | Features can be inspected; SHAP, LIME, tree visualizations. | Harder; need perturbation‑based or attention‑based explanations. |
| **Hardware** | CPUs or cheap GPUs suffice. | GPUs/TPUs often essential; distributed training for very large models. |

---

### 4. When to choose which approach?  

| Problem type | Data characteristics | Ideal technique | Why |
|--------------|----------------------|-----------------|-----|
| **Tabular / structured data** | 50–10 000 features, labeled | Classical ML (GBDT, Random Forest, SVM, Logistic) | Feature engineering possible; interpretability needed; dataset size often moderate. |
| **Low‑sample, high‑dimension data** (e.g., rare disease, scientific labs) | Small number of examples (<1 000) | Shallow ML, possibly with transfer learning or Gaussian Processes | Prevents overfitting; less computational cost. |
| **Natural language** (sentences, documents) | Raw text, large vocab | Transformer‑based models (BERT, GPT) or hierarchical RNNs | End‑to‑end learns syntax & semantics; large corpora help. |
| **Image / video** | Raw pixels, 2‑D arrays | Convolutional neural nets (CNNs), ResNets, Vision Transformers | Exploit spatial locality; need many labeled examples. |
| **Audio / speech** | Raw waveforms or spectrograms | CNN + RNN or pure Transformer (Wav2Vec) | Capture temporal patterns; require large labeled audio. |
| **Time‑series forecasting** | 1‑D sequences, small dataset | Classical ARIMA, Prophet, LSTM (if enough data) | If low data, simple models outperform complex LSTMs. |
| **Sensor / IoT data** | Streaming structured + unstructured | Edge‑optimized CNNs or shallow models with incremental learning | Computational budget on edge devices. |
| **Reinforcement tasks (games, robotics)** | Delayed rewards, high dynamics | Deep RL (DQN, SAC, PPO) | Requires complex policy/value networks to capture state representation. |
| **Generative tasks** | 3‑D data, text, images | GANs, VAEs, diffusion models | Deep nets needed to capture high‑dimensional distribution. |
| **Explainability / regulatory domain** | Healthcare, finance | Classical ML or interpretable DL (e.g., attention, rule‑based post‑hoc) | Regulatory constraints on model opacity. |
| **Limited compute budget** | Sparse or low‑sample data | Classical ML | Easier to train, deploy on CPUs. |

---

### 5. Practical guidelines  

1. **Start with a classical baseline** (logistic regression, random forest).  
   * Gives a quick performance benchmark.  
   * Helps understand feature importance and dataset size needs.  

2. **Profile the data**:  
   * How many samples? Are labels noisy?  
   * Data type: raw images/audio vs structured tables?  

3. **Assess interpretability needs**:  
   * If you must explain decisions, lean toward classical ML or a small interpretable network.  

4. **Check resource constraints**:  
   * GPU time, memory, deployment environment (edge device vs cloud).  

5. **Experiment with transfer learning**:  
   * For vision/text, pretrained models can reduce labeled data needs; fine‑tune a DL model instead of training from scratch.  

6. **Remember that DL is a *subset* of ML**:  
   * A deep network is still a supervised learning algorithm; the same loss functions, regularization techniques, and cross‑validation apply.  

---

### 6. Bottom line

| When to prefer **classic ML** | When to prefer **deep learning** |
|--------------------------------|---------------------------------|
| Structured, tabular data with < 10 k examples | Raw high‑dimensional data (pixels, audio, raw text) with ≥ 10 k labeled examples |
| Need high interpretability and regulatory compliance | End‑to‑end representation learning needed; large data available |
| Limited computation budgets (CPU only) | GPU/TPU resources available |
| Feature engineering is domain‑rich and straightforward | Feature extraction is difficult, expensive, or unknown |
| Small or medium‑size datasets (< 1 000 examples) | Big data, huge corpora (millions of images, billions of tokens) |

In many real‑world projects, the **optimal pipeline** is a *hybrid* approach: start with a robust classical ML model, and if performance plateaus and sufficient data/hardware are available, migrate to a deep learning architecture (often a fine‑tuned pretrained model). This pragmatic stance yields the best of both worlds: solid baselines, interpretability, and the powerful representation learning of deep nets when truly beneficial.

None


In [ ]:
nextQuestionForJudge = f"""
This was the inital question I asked: {question}.

You generated this question for LLMs to answer: {parsedQuestionToBeAskedToLLM}
These are the answers to the question you generated.

Answer from LLM 1:

{gemmaAnswer}

Answer from LLM 2:

{deepseekAnswerCleaned}

Answer from LLM 3:

{gptAnswer}


Now give scores out of 10 based on the answers and rank them. Give in JSON format like {{LLM NAME:  score}}. 
"""

In [43]:
print(nextQuestionForJudge)


This was the inital question I asked: Give me a question that multiple AI models can answer. I want to use the answers from multiple models and score them based on the response they generate. Give me the question only, no answers..

You generated this question for LLMs to answer: What are the key differences between machine learning and deep learning, and in what types of problems is each approach typically more effective?

These are the answers to the question you generated.

Answer from LLM 1:

Okay, let's break down the key differences between machine learning (ML) and deep learning (DL), and then discuss their effectiveness in different problem types.

**1. Machine Learning (ML): The Broad Concept**

* **What it is:** Machine learning is a field of computer science that enables computers to learn from data without being explicitly programmed. It's about creating algorithms that can identify patterns, make predictions, and make decisions based on data.  The algorithm is "trained" o

In [45]:
bodyForJudgement = generateBody(nextQuestionForJudge)

In [ ]:
judgement = generateAnswer(QWEN_MODEL, bodyForJudgement)


In [ ]:
print(display(parseMarkdown(parseResponseByRemovingTag("think", judgement))))